In [1]:
import re
import copy
import numpy as np
from upq import UPQ
from collections.abc import Callable

PROBLEM = 'p01'
PROBLEMS_FILE = 'klotski_problems.txt'

In [2]:
class Block:
    def __init__(self, name: str, x: int, y: int, height: int, width: int):
        self.name = name
        self.x = x
        self.y = y
        self.height = height
        self.width = width

    def __str__(self):
        return f"Block {self.name} - size {self.height}x{self.width} - pos: {self.x}, {self.y}"

class Klotski:
    def __init__(self, problem_id: str):
        with open(PROBLEMS_FILE, 'r', encoding='utf8') as f_in:
            while (line := f_in.readline()).strip() != f'problem {problem_id}' and line:
                pass

            if not line:
                raise KeyError(f'Problem with id "{problem_id}" not found')

            _, n, m = f_in.readline().strip().split(' ')
            self.n, self.m = int(n), int(m)
            self.exit = np.array(f_in.readline().strip().split(' ', maxsplit=2)[-1].split(' '), dtype=int)
            self.target = f_in.readline().strip().split(' ', maxsplit=1)[-1]

            f_in.readline()
            board = list()
            for _ in range(self.n):
                board.append([x.strip() for x in re.split(r'\s+', f_in.readline().strip())])
            self.board = np.array(board)
            self.board_updated = True

            blocks = np.setdiff1d(np.unique(self.board), ['.'])
            self.blocks: dict[str, Block] = dict()
            for block in blocks:
                pos = np.argwhere(self.board == block)
                x, y = pos[0]
                height = np.ptp(pos[:, 0]) + 1
                width = np.ptp(pos[:, 1]) + 1

                self.blocks[str(block)] = Block(block, x, y, height, width)

            self.possible_moves: list[str] = list()
            self.moves_updated = False
    
    def get_board(self) -> np.array:
        if self.board_updated:
            return self.board

        max_len = max(len(b) for b in self.blocks.keys())
        board = np.full((self.n, self.m), '.', dtype=f'<U{max_len}')
        for b in self.blocks.values():
            board[b.x:b.x + b.height, b.y:b.y+b.width] = b.name

        self.board = board
        self.board_updated = True
        return board

    def find_block_moves(self, block: Block) -> list[str]:
        board = self.get_board()
        moves: list[str] = list()
        x, y, h, w = block.x, block.y, block.height, block.width #aliases

        #left
        steps = 1
        can_move = True
        while can_move and y - steps >= 0:
            if np.all(board[x:x+h, y - steps] == '.'):
                moves.append(f"{block.name}-l-{steps}")
                steps += 1
            else:
                can_move = False

        #up
        steps = 1
        can_move = True
        while can_move and x - steps >= 0:
            if np.all(board[x - steps, y:y+w] == '.'):
                moves.append(f"{block.name}-u-{steps}")
                steps += 1
            else:
                can_move = False

        #right
        steps = 1
        can_move = True
        while can_move and y + steps < problem.n:
            if np.all(board[x:x+h, y + steps] == '.'):
                moves.append(f"{block.name}-r-{steps}")
                steps += 1
            else:
                can_move = False

        #down
        steps = 1
        can_move = True
        while can_move and x + steps < problem.m:
            if np.all(board[x + steps, y:y+w] == '.'):
                moves.append(f"{block.name}-d-{steps}")
                steps += 1
            else:
                can_move = False

        #exit
        if np.all(np.isin(np.arange(y, y + w), problem.exit)) and np.all(board[x + 1:, y:y+w] == '.'):
            moves.append(f"{block.name}-e-0")

        return moves

    def find_moves(self) -> list[str]:
        if self.moves_updated:
            return self.possible_moves

        moves: list[str] = list()
        for block in self.blocks.values():
            block_moves = self.find_block_moves(block)
            moves.extend(block_moves)

        self.possible_moves = moves
        self.moves_updated = True
        return moves

    def make_move(self, move: str, create_new: bool = False):
        state = self if not create_new else copy.deepcopy(self)

        block_id, move_type, steps = move.split('-')
        steps = int(steps)

        block = state.blocks.get(block_id)
        if block is None:
            raise KeyError(f'Block in move "{move}" is not on the board')

        valid_moves = state.find_moves()
        if not move in valid_moves:
            raise ValueError(f'The move "{move}" is not valid')

        match move_type:
            case 'e':
                del(state.blocks[block_id])
            case 'u':
                block.x -= steps
            case 'l':
                block.y -= steps
            case 'd':
                block.x += steps
            case 'r':
                block.y += steps
            case _:
                raise ValueError(f'The move {move} has invalid type')

        state.board_updated = False
        state.moves_updated = False

        return state

    def __str__(self):
        max_len = max(len(b) for b in self.blocks.keys())
        board = self.get_board()
        output = f'Size {self.n}x{self.m} - Target: {self.target}\n'
        output += '=' * ((max_len + 1) * self.m +- 1) + '\n'

        for row in board:
            output += (" ".join(s.ljust(max_len) for s in row)) + '\n'

        for i in range(self.m):
            char = ' ' if np.any(self.exit == i) else '='
            mul = max_len + (1 if i != self.m - 1 else 0)
            output += (char * mul)
        
        return output

In [3]:
def goal_state(state: Klotski) -> bool:
    return state.target not in state.blocks

def search(
    initial_state: Klotski,
    goal_test: Callable,
    parent_state: dict,
    parent_move: dict,
    state_cost: dict,
    priority_function: Callable,
    unit_cost: Callable,
) -> list[Klotski]:
    frontier = UPQ()
    parent_state.clear()
    parent_move.clear()
    state_cost.clear()

    state = initial_state
    parent_state[state] = None
    parent_move[state] = None
    state_cost[state] = 0

    while state is not None and not goal_test(state):
        for m in state.find_moves():
            new_state = state.make_move(m, create_new=True)
            cost = unit_cost(m)
            if new_state not in state_cost and new_state not in frontier:
                parent_state[new_state] = state
                parent_move[new_state] = m
                state_cost[new_state] = state_cost[state] + cost
                frontier.push_update(new_state, priority_function(new_state))
            elif new_state in frontier and state_cost[new_state] > state_cost[state] + cost:
                old_cost = state_cost[new_state]
                parent_state[new_state] = state
                parent_move[new_state] = m
                state_cost[new_state] = state_cost[state] + cost
                frontier.push_update(new_state, priority_function(new_state))
        if frontier:
            state, _ = frontier.pop()
        else:
            state = None

    path = list()
    moves = list()
    s = state
    while s:
        path.append(copy.deepcopy(s))
        moves.append(parent_move[s])
        s = parent_state[s]

    return list(reversed(path)), list(reversed(moves))

In [4]:
problem = Klotski(PROBLEM)

print(problem)

Size 3x3 - Target: A
=====
A 5 1
6 4 2
3 . .
==  =


In [5]:
parent_state = dict()
parent_move = dict()
state_cost = dict()

final, moves = search(
    problem,
    goal_test=goal_state,
    parent_state=parent_state,
    parent_move=parent_move,
    state_cost=state_cost,
    priority_function=lambda s: len(state_cost),
    unit_cost=lambda a: 1,
)

for s, m in zip(final, moves):
    if m:
        print(f'Move: {m}')
    print(s, end='\n\n\n')
print(f'Total cost: {len(final) - 1}')

Size 3x3 - Target: A
=====
A 5 1
6 4 2
3 . .
==  =


Move: 4-e-0
Size 3x3 - Target: A
=====
A 5 1
6 . 2
3 . .
==  =


Move: 5-e-0
Size 3x3 - Target: A
=====
A . 1
6 . 2
3 . .
==  =


Move: A-r-1
Size 3x3 - Target: A
=====
. A 1
6 . 2
3 . .
==  =


Move: A-e-0
Size 3x3 - Target: A
=====
. . 1
6 . 2
3 . .
==  =


Total cost: 4
